# 02 Modelling

## Modelling Assumptions and Scope

This model forecasts wholesale electricity prices at the PJM West hub using 
time-based features derived from the trade date: month, day of week, season, 
and similar calendar components. There are no exogenous variables. The model 
learns exclusively from historical price patterns and extrapolates them forward.

The central assumption is that historical calendar patterns are stable enough 
to hold over the forecast horizon. The model has no visibility into fuel prices, 
grid infrastructure changes, policy shifts, or demand shocks such as regional 
growth in data centre capacity. The prediction intervals reflect historical 
price volatility, not the risk of a structural market shift.

A CFO using this output has a historically grounded budget range to work from. 
What lies outside that range requires judgment informed by market expertise, 
procurement conversations, and business context. At Novo Nordisk, building a 
long-term reagent demand forecast for Quality Control reinforced the same 
principle: the value of a forecast is not that it removes uncertainty, but that 
it reduces the uncertainty a decision-maker must reason about themselves.

## Forecasting at Daily Granularity and Aggregating to Monthly

The model produces daily price forecasts rather than monthly ones. This is a deliberate design choice for two reasons.

First, daily data preserves within-month price variation: mid-month spikes, weekly cycles, and holiday effects. Aggregating to monthly before modelling destroys this signal.

Second, the training set contains roughly 5,800 daily observations compared to 192 monthly observations across the same period. More observations means more opportunity to learn seasonal patterns that repeat annually.

### Aggregating prediction intervals to monthly figures

Summing daily interval bounds directly to produce monthly bounds is incorrect. Daily electricity prices are serially correlated: a price spike on one day tends to carry into the next. Summing bounds without accounting for that correlation produces monthly intervals that misrepresent the true uncertainty.

The correct approach is simulation. The model generates 1,000 sample paths, each a plausible sequence of daily prices over the forecast horizon, drawn from the forecast distribution. Each sample path is summed to a monthly total. The 10th and 90th percentiles of those 1,000 monthly totals form the 80% prediction interval. The 2.5th and 97.5th percentiles form the 95% interval.

This propagates uncertainty honestly through the aggregation. The resulting monthly intervals reflect how daily price variability and serial correlation compound over a full month, rather than an assumption that each day's error is independent of the next.

Note: the simulation assumes the model's residuals are a reasonable representation of future forecast errors. The holdout evaluation in `03_holdout_evaluation.ipynb` tests whether that assumption holds in practice.